# Particiones comunes del Entregable 3

Materializa una única división estratificada 70/15/15 con `random_state=42`. Las claves `departamento + fecha_inicio_semana` preservan exactamente la composición usada previamente. Los notebooks experimentales no vuelven a sortear filas.

In [1]:
from pathlib import Path
import hashlib
import json
import pandas as pd
from sklearn.model_selection import train_test_split
ROOT=Path.cwd().parent if Path.cwd().name=="experimentos" else Path.cwd()
RUTA=ROOT/"data/processed/modelado/dataset_modelado_nivel_actividad_firms.parquet"
SALIDA=ROOT/"data/processed/modelado/splits";SALIDA.mkdir(parents=True,exist_ok=True)
datos=pd.read_parquet(RUTA);datos.fecha_inicio_semana=pd.to_datetime(datos.fecha_inicio_semana)
CLAVES=["departamento","fecha_inicio_semana"];TARGET="nivel_actividad_firms";RS=42
assert len(datos)==7923 and not datos.duplicated(CLAVES).any()

In [2]:
itr,temp=train_test_split(datos.index,test_size=.30,random_state=RS,stratify=datos[TARGET])
ival,itest=train_test_split(temp,test_size=.50,random_state=RS,stratify=datos.loc[temp,TARGET])
partes={"train":itr,"validation":ival,"test":itest}
assert set(itr).isdisjoint(ival) and set(itr).isdisjoint(itest) and set(ival).isdisjoint(itest)
assert sum(map(len,partes.values()))==len(datos)

In [3]:
def hash_claves(tabla):return hashlib.sha256(tabla.sort_values(CLAVES)[CLAVES].to_csv(index=False).encode()).hexdigest()
meta={"random_state":RS,"target":TARGET,"estratificado":True,"dataset":str(RUTA.relative_to(ROOT)),"particiones":{}}
for nombre,indices in partes.items():
 tabla=datos.loc[indices,CLAVES].sort_values(CLAVES).reset_index(drop=True);tabla.to_parquet(SALIDA/f"{nombre}_keys.parquet",index=False)
 meta["particiones"][nombre]={"filas":len(tabla),"sha256_claves":hash_claves(tabla),"distribucion":datos.loc[indices,TARGET].value_counts().to_dict()}
(SALIDA/"splits_metadata.json").write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding="utf-8")
meta

{'random_state': 42,
 'target': 'nivel_actividad_firms',
 'estratificado': True,
 'dataset': 'data/processed/modelado/dataset_modelado_nivel_actividad_firms.parquet',
 'particiones': {'train': {'filas': 5546,
   'sha256_claves': '9f3ae5eac78a5b69ca8ec833ee739bb7d5db60f3ed357cc4a268bf558007d944',
   'distribucion': {'Sin actividad': 3477,
    'Moderado': 886,
    'Bajo': 866,
    'Alto': 317}},
  'validation': {'filas': 1188,
   'sha256_claves': '001ff6c54707051e4325e13d226fd9bfb8eb67f5f595017d8760a5ff03cfe3f2',
   'distribucion': {'Sin actividad': 745,
    'Moderado': 190,
    'Bajo': 185,
    'Alto': 68}},
  'test': {'filas': 1189,
   'sha256_claves': 'cbd42ce416cb136d9cda483e1898677078aca3c0ad43622afd999746cfc51e76',
   'distribucion': {'Sin actividad': 745,
    'Moderado': 190,
    'Bajo': 186,
    'Alto': 68}}}}

Los archivos de test contienen únicamente claves. Ningún notebook experimental los utiliza para métricas o selección; sólo `Comparacion_Modelos.ipynb` los consume después de congelar la solución.